# Customer Intelligence System — Step 1: Load & Clean
Goal: keep only real purchases by known customers. Order matters: types → missing IDs → duplicates → non-product codes → net cancellations → bad prices.

In [1]:
# 1.1 Paths (notebook lives in notebooks/, data in data/)
import pandas as pd, numpy as np
from pathlib import Path

RAW  = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed"); PROC.mkdir(parents=True, exist_ok=True)
print("Raw file found:", RAW.exists())

Raw file found: True


In [2]:
# 1.2 Load (works for both the Excel and CSV versions)
path = RAW
if path.suffix == ".xlsx":
    sheets = pd.read_excel(path, sheet_name=None)       # reads every sheet (one per year)
    df = pd.concat(sheets.values(), ignore_index=True)  # takes 1-2 min, normal
else:
    df = pd.read_csv(path, encoding="ISO-8859-1")

df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# 1.3 Fix types, drop missing customers, drop duplicates
def step(name, d):
    print(f"{name:<32}{len(d):>10,} rows")
    return d

df["Invoice"]     = df["Invoice"].astype(str)
df["StockCode"]   = df["StockCode"].astype(str).str.strip().str.upper()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df = step("raw", df)

df = df.dropna(subset=["Customer ID"]).copy()
df["Customer ID"] = df["Customer ID"].astype(int)
df = step("after dropping missing IDs", df)

df = df.drop_duplicates()   # done early so a duplicated purchase can't be matched twice later
df = step("after dropping duplicates", df)

raw                              1,067,371 rows
after dropping missing IDs         824,364 rows
after dropping duplicates          797,885 rows


In [4]:
# 1.4 Remove non-product codes (postage, fees, manual adjustments, tests)
# Real products look like 5 digits + optional letters, e.g. 85123A
is_product = df["StockCode"].str.match(r"^\d{5}[A-Z]{0,2}$")

print("Codes being removed (check nothing real is in here):")
print(df.loc[~is_product, "StockCode"].value_counts().head(20))

df = df[is_product]
df = step("after removing non-products", df)

Codes being removed (check nothing real is in here):
StockCode
POST            1983
M               1085
C2               254
D                170
ADJUST            61
BANK CHARGES      37
PADS              19
DOT               16
CRUK              16
TEST001           15
ADJUST2            3
SP1002             2
TEST002            1
Name: count, dtype: int64
after removing non-products        794,223 rows


In [5]:
# 1.5 Net cancellations against their original purchase
# Deleting only the 'C' rows leaves the original order in -> fake big spenders.
# Rule: for each cancellation, find the latest earlier purchase by the same customer,
# same product, same quantity -> remove both.
is_cancel = df["Invoice"].str.startswith("C")
cancels   = df[is_cancel].sort_values("InvoiceDate")
purchases = df[~is_cancel].sort_values("InvoiceDate")

lookup = {}   # (customer, product) -> list of (date, qty, row index), oldest first
for idx, cid, sc, dt, q in zip(purchases.index, purchases["Customer ID"], purchases["StockCode"],
                               purchases["InvoiceDate"], purchases["Quantity"]):
    lookup.setdefault((cid, sc), []).append((dt, q, idx))

to_drop = set()
for cid, sc, dt, q in zip(cancels["Customer ID"], cancels["StockCode"],
                          cancels["InvoiceDate"], cancels["Quantity"]):
    for p_dt, p_q, p_idx in reversed(lookup.get((cid, sc), [])):
        if p_dt <= dt and p_q == -q and p_idx not in to_drop:
            to_drop.add(p_idx)
            break

print(f"Cancellation lines: {len(cancels):,} | matched to an original: {len(to_drop):,}")
purchases = purchases.drop(index=list(to_drop))

# keep cancellations aside -> becomes the 'return rate' feature later
cancels.to_parquet(PROC / "cancellations.parquet")

Cancellation lines: 17,586 | matched to an original: 6,228


In [6]:
# 1.6 Drop remaining bad rows, add revenue
df = purchases[(purchases["Quantity"] > 0) & (purchases["Price"] > 0)].copy()
df["Revenue"] = df["Quantity"] * df["Price"]
df = step("final clean purchases", df)

final clean purchases              770,350 rows


In [7]:
# 1.7 Sanity checks
print("Customers :", df["Customer ID"].nunique())
print("Invoices  :", df["Invoice"].nunique())
print("Dates     :", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("\nTop 5 customers by revenue (should look plausible, no 80K-unit ghosts):")
print(df.groupby("Customer ID")["Revenue"].sum().sort_values(ascending=False).head())
print("\nTop countries (% of rows):")
print((df["Country"].value_counts(normalize=True).head(5) * 100).round(1))

Customers : 5839
Invoices  : 36324
Dates     : 2009-12-01 07:45:00 -> 2011-12-09 12:50:00

Top 5 customers by revenue (should look plausible, no 80K-unit ghosts):
Customer ID
18102    579128.64
14646    523361.34
14156    298378.79
14911    259839.13
17450    233675.91
Name: Revenue, dtype: float64

Top countries (% of rows):
Country
United Kingdom    90.1
Germany            2.0
EIRE               2.0
France             1.7
Netherlands        0.6
Name: proportion, dtype: float64


In [8]:
# 1.8 Save so we never re-read the slow raw file
df.to_parquet(PROC / "clean_retail.parquet", index=False)
print("saved clean_retail.parquet")

saved clean_retail.parquet
